# Aufgabe 2: CNN zur Auto-Erkennung

Zur Erkennung von mehreren Autos auf Bildern werden 2 Ansätze verfolgt:
1. Verwendung des vortrainierten Netzes mit einem Sliding Window
2. Verwendung des vortrainierten Netzes mit Detection-Algorithmus YOLOv8

In [1]:
# Imports

import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import numpy as np
import cv2


In [ ]:
# Laden des Models

current_dir = Path.cwd()
model_path = current_dir.parent / "models" / "task1a" / "car_cnn.keras"
model = keras.models.load_model(model_path)

c:\Users\hcr9bh\T2000\Projekt-Lernverfahren\lernverfahren\models\task1a\car_cnn.keras


1 Vortrainiertes Netz mit Sliding Window

In [3]:
# sliding window
def sliding_window(image, step_size, window_size):
    window_width = window_size[0]
    window_height = window_size[1]
    image_height, image_width = image.shape[:2]

    # loop over image
    for y in range (0, image_height, step_size):
        for x in range (0, image_width, step_size):
            window = image[y : y + window_height, x : x + window_width]
            
            if (window.shape[:2] != (window_height, window_width)) :
                #hier umgang mit unvollständigen Windows definieren -> z. B. Padding
                continue
            yield (x, y, window)




def preprocess_window(window):
    """Preprocessing wie CIFAR-10"""

    if window.shape[:2] != (32, 32):
        window = cv2.resize(window, (32, 32))
    
    window_rgb = cv2.cvtColor(window, cv2.COLOR_BGR2RGB)
    window_normalized = window_rgb.astype(np.float32) / 255.0
    
    return window_normalized



# # Anzeige des sliding windows
# image = cv2.imread("../src/images/2cars_640px.jpg")
# window_size = (64, 64)

# for (x, y, window) in sliding_window(image, 32, window_size):
#     clone = image.copy()
#     cv2.rectangle(clone, (x, y), (x + window_size[0], y + window_size[1]), (0, 255, 0), 2 )
#     cv2.imshow("Car", clone)
#     cv2.waitKey(100)
    
#     #prediciton
#     processed_window = preprocess_window(window)
#     input_batch = np.expand_dims(processed_window, axis=0)
#     prediction = model.predict(input_batch, verbose=0)
#     confidence = prediction[0][0] * 100
#     if confidence > 50:
#         print(confidence, "coord x:", x, "coord y:", y)


# cv2.destroyAllWindows()


In [4]:
# sliding window batch
def sliding_window_batch(image, step_size, window_size, model, threshold):
    window_width = window_size[0]
    window_height = window_size[1]
    image_height, image_width = image.shape[:2]
    windows = []
    positions = []

   # Select Windows
    for y in range (0, image_height, step_size):
        for x in range (0, image_width, step_size):
            window = image[y : y + window_height, x : x + window_width]
            
            if (window.shape[:2] == (window_height, window_width)) :
                windows.append(preprocess_window(window))
                positions.append((x, y))
                            
    # Batch prediction
    windows_array = np.array(windows)
    predictions = model.predict(windows_array, batch_size=64, verbose=0)

    # Collection relevant results
    detections = []
    for (x, y), pred in zip(positions, predictions):
        confidence = pred[0]
        if confidence > threshold:
            detections.append({
                'x_start': x, 
                'y_start': y, 
                'x_end': x + window_width,
                'y_end': y + window_height,
                'confidence': float(confidence)
            })
    
    return detections



In [5]:
# Non maximum surpression

def compute_iou(box1, box2):
    """
    Berechnet Intersection over Union (IoU)
    """
    # Überlappungsbereich
    x1 = max(box1['x_start'], box2['x_start'])
    y1 = max(box1['y_start'], box2['y_start'])
    x2 = min(box1['x_end'], box2['x_end'])
    y2 = min(box1['y_end'], box2['y_end'])
    
    # Keine Überlappung
    if x2 < x1 or y2 < y1:
        return 0.0
    
    # Flächen
    intersection = (x2 - x1) * (y2 - y1)
    area1 = (box1['x_end'] - box1['x_start']) * (box1['y_end'] - box1['y_start'])
    area2 = (box2['x_end'] - box2['x_start']) * (box2['y_end'] - box2['y_start'])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0

def non_max_suppression(detections, overlap_thresh=0.3):
    """
    Entfernt überlappende Detektionen
    
    overlap_thresh: Maximale erlaubte Überlappung (IoU)
    """
    if len(detections) == 0:
        return []
    
    # Nach Confidence sortieren (höchste zuerst)
    detections = sorted(detections, key=lambda x: x['confidence'], reverse=True)
    
    keep = []
    
    while len(detections) > 0:
        # Beste Detection übernehmen
        best = detections.pop(0)
        keep.append(best)
        
        # Überlappende Detections entfernen
        detections = [
            det for det in detections 
            if compute_iou(best, det) < overlap_thresh
        ]
    
    return keep

In [18]:
# image pyramid

def image_pyramid (image, scales = [1.5, 1.25, 1, 0.75, 0.5, 0.25]):
    """
        Erstellt Bildpyramide mit mehreren Skalen
        Returns: Liste von (scaled_image, scale_factor)
    """
    pyramid = []
    
    for scale in scales:
        new_width = int(image.shape[1] * scale)
        new_height = int(image.shape[0] * scale)
        
        # Überspringe zu kleine Bilder
        if new_width < 32 or new_height < 32:
            continue
            
        scaled = cv2.resize(image, (new_width, new_height))
        pyramid.append((scaled, scale))
    
    return pyramid

In [ ]:
## Complete Algorithm

#=== PARAMETER ===
STEP_SIZE = 8
WINDOW_SIZE = (32, 32)
CONFIDENCE_THRESHOLD = 0.7
NMS_THRESHOLD = 0.4

# 1. read image
image = cv2.imread("../src/images/2cars_640px.jpg")

# 2. create images with image pyramid
scaled_images = image_pyramid(image)

# 3. Über Skalen iterien
all_detections = []

for scaled_im, scale_factor in scaled_images:
    print(f"Imagesize: {scaled_im.shape[0]} und {scaled_im.shape[1]}")
    detections = sliding_window_batch(
        scaled_im, 
        step_size=STEP_SIZE, 
        window_size=WINDOW_SIZE, 
        model=model, 
        threshold=CONFIDENCE_THRESHOLD)
    
    print(f"  Window {WINDOW_SIZE}: {len(detections)} Detektionen")
        
    # Koordinaten zurück auf Original skalieren
    for det in detections:
        det['x_start'] = int(det['x_start'] / scale_factor)
        det['y_start'] = int(det['y_start'] / scale_factor)
        det['x_end'] = int(det['x_end'] / scale_factor)
        det['y_end'] = int(det['y_end'] / scale_factor)
        det['scale'] = scale_factor
        det['window_size'] = WINDOW_SIZE

    all_detections.extend(detections)

print(f"\nGesamt vor NMS: {len(all_detections)} Detektionen")

# 4. NMS über alle Detektionen
detections_filtered = non_max_suppression(all_detections, overlap_thresh=NMS_THRESHOLD)
print(f"\nGesamt nach NMS: {len(detections_filtered)} Detektionen")

# 5. Visualisierung
result = image.copy()

for det in detections_filtered:
    # Bounding Box
    cv2.rectangle(result, 
                  (det['x_start'], det['y_start']), 
                  (det['x_end'], det['y_end']), 
                  (0, 255, 0), 2)
    
    # Label mit Confidence und Skala
    label = f"{det['confidence']:.2f} @{det['scale']:.1f}x"
    cv2.putText(result, label, 
                (det['x_start'], det['y_start'] - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

cv2.imshow("Car Detection", result)
cv2.waitKey(0)
cv2.destroyAllWindows()
    


Imagesize: 640 und 960
  Window (32, 32): 59 Detektionen
Imagesize: 533 und 800
  Window (32, 32): 59 Detektionen
Imagesize: 427 und 640
  Window (32, 32): 52 Detektionen
Imagesize: 320 und 480
  Window (32, 32): 31 Detektionen
Imagesize: 213 und 320
  Window (32, 32): 20 Detektionen
Imagesize: 106 und 160
  Window (32, 32): 6 Detektionen

Gesamt vor NMS: 227 Detektionen

Gesamt nach NMS: 79 Detektionen
